In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 1. Hardware Verification & Hardware Acceleration Setup
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Ensure you are using your rented A100 GPU!"
model_id = "google/gemma-4-E2B"

# 2. Load Tokenizer and Dataset
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Base models need a padding token defined

dataset = load_dataset("json", data_files={"train": "it-by-stephen-king.jsonl"})

# 3. Load the Base Model in Full bfloat16 Precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 4. Define target modules and configure LoRA Adapters
peft_config = LoraConfig(
    r=64,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",

    # key difference vs your TRL run
    use_rslora=True,

    target_modules=[
        "q_proj.linear",
        "k_proj.linear",
        "v_proj.linear",
        "o_proj.linear",
        "gate_proj.linear",
        "up_proj.linear",
        "down_proj.linear",
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. Define Training Hyperparameters
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./gemma4-lora",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=1,

    learning_rate=5e-5,

    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    weight_decay=0.01,

    bf16=True,

    logging_steps=10,
    save_strategy="epoch",
    report_to="none",

    max_length=4096,

    # THIS is your Unsloth "packing=True"
    packing=True,

    gradient_checkpointing=True,
)

# 6. Initialize SFTTrainer and run training loop
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

print("Starting LoRA fine-tuning loop...")
trainer.train()

# 7. Save the trained adapter weights (not the whole model)
trainer.model.save_pretrained("./final_lora_adapter")
tokenizer.save_pretrained("./final_lora_adapter")
print("Training complete! Adapter weights saved.")
